<div dir="rtl" style="text-align: right; line-height: 1.9; font-family: Tahoma, Arial, sans-serif; font-size: 15px;">

<h2>تمرین دوم کدی یادگیری ماشین</h2>

<p>
هدف این تمرین، تشخیص مثبت یا منفی بودن نتیجه کرونا با استفاده از داده‌های افراد مختلفه. مدل اصلی به صورت دستی با <span dir="ltr">Numpy</span> پیاده‌سازی شده و بعد نتیجه آن با مدل آماده <span dir="ltr">Logistic Regression</span> در <span dir="ltr">Scikit-Learn</span> مقایسه میشه.
</p>

<p>
در طول تمرین، داده‌های ناقص پر میشن، ستون‌های متنی به ویژگی عددی تبدیل میشن و تقسیم داده‌ها به صورت ثابت انجام میشه. برای پارامترهای آزاد، مقدارهایی انتخاب شدن که نه خیلی خاص باشن و نه رفتار مدل رو غیرطبیعی کنن؛ مثلا سه مقدار <span dir="ltr">0.015</span>، <span dir="ltr">0.04</span> و <span dir="ltr">0.065</span> برای <span dir="ltr">Learning Rate</span> بررسی شدن.
</p>

</div>

In [ ]:
import csv
import importlib
import numpy as np
import matplotlib.pyplot as plt

from IPython.display import HTML, display
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression as SklearnLogisticRegression

import LogisticRegression as custom_lr_module
importlib.reload(custom_lr_module)
from LogisticRegression import LogisticRegression


DATA_PATH = "Covid_pula_pula_bompa.csv"
TARGET_COLUMN = "CovidPos"

TEST_SIZE = 0.20
RANDOM_STATE = 34

LEARNING_RATES = [0.015, 0.04, 0.065]
EPOCHS = 160

MAIN_LEARNING_RATE = 0.04

TARGET_CORR_THRESHOLD = 0.02
PAIRWISE_CORR_THRESHOLD = 0.92
MIN_SELECTED_FEATURES = 35
MAX_SELECTED_FEATURES = 65

MISSING_VALUES = {"", " ", "nan", "none", "null", "na", "n/a", "missing"}


def read_csv_file(path):
    with open(path, "r", encoding="utf-8-sig", newline="") as file:
        reader = csv.reader(file)
        header = next(reader)
        rows = []
        skipped_rows = 0

        for row in reader:
            if len(row) == len(header):
                rows.append(row)
            elif len(row) < len(header):
                rows.append(row + [""] * (len(header) - len(row)))
            else:
                rows.append(row[:len(header)])
                skipped_rows += 1

    return header, np.array(rows, dtype=object), skipped_rows


header, data, skipped_rows = read_csv_file(DATA_PATH)

target_index = header.index(TARGET_COLUMN)
target_text = np.array([str(value).strip().lower() for value in data[:, target_index]], dtype=object)
valid_target_mask = np.isin(target_text, ["yes", "no"])

y = (target_text[valid_target_mask] == "yes").astype(np.int32)
X_raw = data[valid_target_mask][:, [i for i in range(len(header)) if i != target_index]]
feature_columns = [name for i, name in enumerate(header) if i != target_index]

indices = np.arange(len(y))
train_idx, test_idx = train_test_split(
    indices,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y
)

X_train_raw = X_raw[train_idx]
X_test_raw = X_raw[test_idx]
y_train = y[train_idx]
y_test = y[test_idx]

print("Raw rows:", data.shape[0])
print("Used rows:", len(y))
print("Skipped rows because of extra columns:", skipped_rows)
print("Train rows:", len(y_train))
print("Test rows:", len(y_test))
print("Positive class ratio in train:", round(float(np.mean(y_train)), 4))
print("Positive class ratio in test:", round(float(np.mean(y_test)), 4))

<div dir="rtl" style="text-align: right; line-height: 1.9; font-family: Tahoma, Arial, sans-serif; font-size: 15px;">

<h3>۱. آماده‌سازی داده‌ها</h3>

<p>
در این بخش، فایل <span dir="ltr">Covid_pula_pula_bompa.csv</span> خوانده میشه و ستون هدف یعنی <span dir="ltr">CovidPos</span> از بقیه ستون‌ها جدا میشه. فقط رکوردهایی نگه داشته میشن که مقدار هدف آن‌ها مشخص باشه، چون برای آموزش مدل باید جواب درست هر نمونه را داشته باشیم.
</p>

<p>
بعد از جدا کردن داده‌ها، تقسیم آموزش و تست با نسبت <span dir="ltr">80/20</span> انجام میشه. برای اینکه نسبت کلاس‌های مثبت و منفی در دو بخش خیلی به‌هم نخوره، از <span dir="ltr">stratify</span> استفاده شده. مقدارهای خالی ستون‌های عددی با میانه و مقدارهای خالی ستون‌های متنی با مقدار <span dir="ltr">Missing</span> پر میشن. در آخر، ستون‌های متنی با <span dir="ltr">One-Hot Encoding</span> عددی میشن و ویژگی‌های عددی هم استاندارد میشن.
</p>

</div>

In [ ]:
def clean_text(value):
    return str(value).strip()


def is_missing(value):
    return clean_text(value).lower() in MISSING_VALUES


def try_float(value):
    if is_missing(value):
        return np.nan

    try:
        return float(clean_text(value))
    except ValueError:
        return np.nan


def detect_column_types(X, columns, numeric_ratio=0.95):
    numeric_indices = []
    categorical_indices = []

    for col_idx, _ in enumerate(columns):
        values = X[:, col_idx]
        non_missing = [value for value in values if not is_missing(value)]

        if len(non_missing) == 0:
            categorical_indices.append(col_idx)
            continue

        converted_count = 0
        for value in non_missing:
            if not np.isnan(try_float(value)):
                converted_count += 1

        if converted_count / len(non_missing) >= numeric_ratio:
            numeric_indices.append(col_idx)
        else:
            categorical_indices.append(col_idx)

    return numeric_indices, categorical_indices


def to_float_column(values):
    result = np.empty(len(values), dtype=np.float32)

    for i, value in enumerate(values):
        result[i] = try_float(value)

    return result


def fit_preprocessor(X_train_raw, columns):
    numeric_indices, categorical_indices = detect_column_types(X_train_raw, columns)

    numeric_info = {}
    for idx in numeric_indices:
        values = to_float_column(X_train_raw[:, idx])
        median = np.nanmedian(values)
        values = np.where(np.isnan(values), median, values)

        mean = np.mean(values)
        std = np.std(values)
        if std == 0:
            std = 1.0

        numeric_info[idx] = {
            "median": float(median),
            "mean": float(mean),
            "std": float(std),
        }

    categorical_info = {}
    for idx in categorical_indices:
        values = np.array([
            "Missing" if is_missing(value) else clean_text(value)
            for value in X_train_raw[:, idx]
        ], dtype=object)

        categories = np.array(sorted(np.unique(values)), dtype=object)

        if len(categories) > 1:
            categories_to_encode = categories[1:]
        else:
            categories_to_encode = categories

        categorical_info[idx] = {
            "categories": categories_to_encode,
            "mapping": {category: i for i, category in enumerate(categories_to_encode)},
        }

    return {
        "columns": columns,
        "numeric_indices": numeric_indices,
        "categorical_indices": categorical_indices,
        "numeric_info": numeric_info,
        "categorical_info": categorical_info,
    }


def transform_data(X_raw, preprocessor):
    processed_parts = []
    feature_names = []

    for idx in preprocessor["numeric_indices"]:
        values = to_float_column(X_raw[:, idx])
        info = preprocessor["numeric_info"][idx]

        values = np.where(np.isnan(values), info["median"], values)
        values = (values - info["mean"]) / info["std"]

        processed_parts.append(values.reshape(-1, 1).astype(np.float32))
        feature_names.append(preprocessor["columns"][idx])

    for idx in preprocessor["categorical_indices"]:
        values = np.array([
            "Missing" if is_missing(value) else clean_text(value)
            for value in X_raw[:, idx]
        ], dtype=object)

        info = preprocessor["categorical_info"][idx]
        categories = info["categories"]
        mapping = info["mapping"]

        encoded = np.zeros((len(values), len(categories)), dtype=np.float32)
        category_indices = np.array([mapping.get(value, -1) for value in values], dtype=np.int32)

        valid_mask = category_indices >= 0
        encoded[np.arange(len(values))[valid_mask], category_indices[valid_mask]] = 1.0

        processed_parts.append(encoded)

        for category in categories:
            feature_names.append(f"{preprocessor['columns'][idx]}={category}")

    X_processed = np.hstack(processed_parts).astype(np.float32)
    return X_processed, feature_names


preprocessor = fit_preprocessor(X_train_raw, feature_columns)

X_train, feature_names = transform_data(X_train_raw, preprocessor)
X_test, _ = transform_data(X_test_raw, preprocessor)

print("Final train shape:", X_train.shape)
print("Final test shape:", X_test.shape)
print("Numeric columns:", len(preprocessor["numeric_indices"]))
print("Categorical columns:", len(preprocessor["categorical_indices"]))
print("Final number of features:", len(feature_names))

<div dir="rtl" style="text-align: right; line-height: 1.9; font-family: Tahoma, Arial, sans-serif; font-size: 15px;">

<h3>۲. توابع ارزیابی مدل</h3>

<p>
برای مقایسه مدل‌ها، معیارهای <span dir="ltr">Accuracy</span>، <span dir="ltr">Precision</span>، <span dir="ltr">Recall</span> و <span dir="ltr">AUC</span> حساب میشن. این معیارها از روی تعداد نمونه‌های درست و غلط در دو کلاس مثبت و منفی به دست میان.
</p>

<p>
همچنین برای رسم <span dir="ltr">ROC Curve</span>، آستانه تصمیم‌گیری از ۱ تا ۰ تغییر داده میشه و در هر آستانه مقدارهای <span dir="ltr">TPR</span> و <span dir="ltr">FPR</span> محاسبه میشن. این کار باعث میشه عملکرد مدل فقط به آستانه ثابت <span dir="ltr">0.5</span> وابسته نباشه.
</p>

</div>

In [ ]:
def confusion_values(y_true, y_pred):
    y_true = np.asarray(y_true).astype(int)
    y_pred = np.asarray(y_pred).astype(int)

    tp = int(np.sum((y_true == 1) & (y_pred == 1)))
    tn = int(np.sum((y_true == 0) & (y_pred == 0)))
    fp = int(np.sum((y_true == 0) & (y_pred == 1)))
    fn = int(np.sum((y_true == 1) & (y_pred == 0)))

    return tp, tn, fp, fn


def roc_curve_manual(y_true, y_score, points=101):
    thresholds = np.linspace(1.0, 0.0, points)
    fpr_values = []
    tpr_values = []

    for threshold in thresholds:
        y_pred = (y_score >= threshold).astype(int)
        tp, tn, fp, fn = confusion_values(y_true, y_pred)

        tpr = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0

        tpr_values.append(tpr)
        fpr_values.append(fpr)

    return np.array(fpr_values), np.array(tpr_values)


def auc_score(fpr, tpr):
    order = np.argsort(fpr)
    return float(np.trapz(tpr[order], fpr[order]))


def evaluate_model(y_true, y_pred, y_score):
    tp, tn, fp, fn = confusion_values(y_true, y_pred)

    accuracy = (tp + tn) / len(y_true)
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0

    fpr, tpr = roc_curve_manual(y_true, y_score)
    auc = auc_score(fpr, tpr)

    return {
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "AUC": auc,
    }


def display_table(rows, columns, title=None):
    html = '<div dir="ltr" style="font-family: Arial, sans-serif;">'

    if title is not None:
        html += f"<h4>{title}</h4>"

    html += '<table style="border-collapse: collapse; font-size: 14px;">'
    html += "<tr>"

    for column in columns:
        html += f'<th style="border: 1px solid #999; padding: 6px 10px;">{column}</th>'

    html += "</tr>"

    for row in rows:
        html += "<tr>"
        for column in columns:
            value = row[column]
            if isinstance(value, float):
                value = f"{value:.4f}"
            html += f'<td style="border: 1px solid #999; padding: 6px 10px;">{value}</td>'
        html += "</tr>"

    html += "</table></div>"
    display(HTML(html))

<div dir="rtl" style="text-align: right; line-height: 1.9; font-family: Tahoma, Arial, sans-serif; font-size: 15px;">

<h3>۳. بخش الف: پیاده‌سازی <span dir="ltr">Logistic Regression</span> با <span dir="ltr">Gradient Descent</span></h3>

<p>
در مدل دستی، ابتدا ترکیب خطی ویژگی‌ها با وزن‌ها ساخته میشه و بعد خروجی آن وارد تابع <span dir="ltr">sigmoid</span> میشه. خروجی <span dir="ltr">sigmoid</span> یک عدد بین صفر و یکه و به عنوان احتمال مثبت بودن کلاس در نظر گرفته میشه.
</p>

<p>
برای تابع خطا از <span dir="ltr">Binary Cross Entropy</span> استفاده شده، چون مسئله فقط دو کلاس داره. گرادیان وزن‌ها و بایاس در فایل <span dir="ltr">LogisticRegression.py</span> به صورت دستی حساب شده و از <span dir="ltr">autograd</span> استفاده نشده. در این قسمت مدل با چند مقدار مختلف برای <span dir="ltr">Learning Rate</span> آموزش داده میشه تا اثر آن روی کاهش خطا و معیارهای مدل مشخص بشه.
</p>

</div>

In [ ]:
custom_models = {}
part_a_rows = []

plt.figure(figsize=(8, 5))

for learning_rate in LEARNING_RATES:
    model = LogisticRegression()
    model.fit(X_train, y_train, epochs=EPOCHS, learning_rate=learning_rate)

    y_score = model.predict_proba(X_test)
    y_pred = model.predict(X_test)
    metrics = evaluate_model(y_test, y_pred, y_score)

    custom_models[learning_rate] = model

    part_a_rows.append({
        "Learning Rate": learning_rate,
        "Accuracy": metrics["Accuracy"],
        "Precision": metrics["Precision"],
        "Recall": metrics["Recall"],
        "AUC": metrics["AUC"],
        "Final Loss": model.loss_history[-1],
    })

    plt.plot(model.loss_history, label=f"lr={learning_rate}")

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Loss Curve for Different Learning Rates")
plt.legend()
plt.grid(True)
plt.show()

display_table(
    part_a_rows,
    ["Learning Rate", "Accuracy", "Precision", "Recall", "AUC", "Final Loss"],
    "Part A - Custom Logistic Regression"
)

plt.figure(figsize=(7, 5))

for learning_rate, model in custom_models.items():
    y_score = model.predict_proba(X_test)
    fpr, tpr = roc_curve_manual(y_test, y_score)
    auc = auc_score(fpr, tpr)
    plt.plot(fpr, tpr, label=f"lr={learning_rate}, AUC={auc:.3f}")

plt.plot([0, 1], [0, 1], linestyle="--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve for Custom Models")
plt.legend()
plt.grid(True)
plt.show()

<div dir="rtl" style="text-align: right; line-height: 1.9; font-family: Tahoma, Arial, sans-serif; font-size: 15px;">

<p>
طبق نمودار خطا، تغییر مقدار <span dir="ltr">Learning Rate</span> روی سرعت کاهش <span dir="ltr">Loss</span> اثر مستقیم داره. مقدارهای خیلی کوچک معمولا مدل را آهسته‌تر جلو می‌برن و مقدارهای بزرگ‌تر ممکنه نوسان بیشتری ایجاد کنن. برای ادامه مقایسه‌ها، مقدار میانی <span dir="ltr">0.04</span> انتخاب شده، چون مقدار متعادلیه و رفتار مدل را خیلی کند یا خیلی ناپایدار نمی‌کنه.
</p>

</div>

<div dir="rtl" style="text-align: right; line-height: 1.9; font-family: Tahoma, Arial, sans-serif; font-size: 15px;">

<h3>۴. بخش ب: مقایسه با مدل <span dir="ltr">Scikit-Learn</span></h3>

<p>
در این بخش، مدل دستی با مدل آماده <span dir="ltr">Logistic Regression</span> مقایسه میشه. برای اینکه مقایسه منصفانه باشه، هر دو مدل با همان داده‌های آموزش و تست بخش قبل اجرا میشن و معیارهای ارزیابی هم دقیقا با توابع یکسان محاسبه میشن.
</p>

<p>
مدل آماده <span dir="ltr">Scikit-Learn</span> از روش‌های بهینه‌سازی آماده‌تر استفاده می‌کنه، ولی هدف این بخش اینه که ببینیم پیاده‌سازی دستی تا چه حد به نتیجه مدل آماده نزدیک میشه.
</p>

</div>

In [ ]:
custom_main_model = custom_models[MAIN_LEARNING_RATE]
custom_score = custom_main_model.predict_proba(X_test)
custom_pred = custom_main_model.predict(X_test)
custom_metrics = evaluate_model(y_test, custom_pred, custom_score)

try:
    sklearn_model = SklearnLogisticRegression(
        max_iter=700,
        solver="lbfgs",
        penalty=None
    )
    sklearn_model.fit(X_train, y_train)
except Exception:
    sklearn_model = SklearnLogisticRegression(
        max_iter=700,
        solver="lbfgs",
        penalty="none"
    )
    sklearn_model.fit(X_train, y_train)

sklearn_pred = sklearn_model.predict(X_test)
sklearn_score = sklearn_model.predict_proba(X_test)[:, 1]
sklearn_metrics = evaluate_model(y_test, sklearn_pred, sklearn_score)

part_b_rows = [
    {
        "Model": "Custom Logistic Regression",
        "Accuracy": custom_metrics["Accuracy"],
        "Precision": custom_metrics["Precision"],
        "Recall": custom_metrics["Recall"],
        "AUC": custom_metrics["AUC"],
    },
    {
        "Model": "Sklearn Logistic Regression",
        "Accuracy": sklearn_metrics["Accuracy"],
        "Precision": sklearn_metrics["Precision"],
        "Recall": sklearn_metrics["Recall"],
        "AUC": sklearn_metrics["AUC"],
    },
]

display_table(
    part_b_rows,
    ["Model", "Accuracy", "Precision", "Recall", "AUC"],
    "Part B - Custom Model vs Sklearn Model"
)

plt.figure(figsize=(7, 5))

custom_fpr, custom_tpr = roc_curve_manual(y_test, custom_score)
sklearn_fpr, sklearn_tpr = roc_curve_manual(y_test, sklearn_score)

plt.plot(custom_fpr, custom_tpr, label=f"Custom, AUC={auc_score(custom_fpr, custom_tpr):.3f}")
plt.plot(sklearn_fpr, sklearn_tpr, label=f"Sklearn, AUC={auc_score(sklearn_fpr, sklearn_tpr):.3f}")
plt.plot([0, 1], [0, 1], linestyle="--")

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve: Custom vs Sklearn")
plt.legend()
plt.grid(True)
plt.show()

<div dir="rtl" style="text-align: right; line-height: 1.9; font-family: Tahoma, Arial, sans-serif; font-size: 15px;">

<p>
اختلاف نتیجه دو مدل طبیعی‌ه، چون مدل دستی با <span dir="ltr">Gradient Descent</span> ساده و تعداد تکرار مشخص آموزش داده شده، اما مدل <span dir="ltr">Scikit-Learn</span> تنظیمات و روش بهینه‌سازی کامل‌تری داره. با این حال، چون داده و معیارهای ارزیابی برای هر دو مدل یکیه، میشه عملکرد آن‌ها را مستقیم با هم مقایسه کرد.
</p>

</div>

<div dir="rtl" style="text-align: right; line-height: 1.9; font-family: Tahoma, Arial, sans-serif; font-size: 15px;">

<h3>۵. بخش ج: کاهش ویژگی‌ها با <span dir="ltr">Correlation Matrix</span></h3>

<p>
در این قسمت بررسی میشه که آیا همه ویژگی‌ها برای آموزش مدل لازم هستن یا نه. اول ارتباط هر ویژگی با خروجی محاسبه میشه و بعد هم ارتباط خطی بین خود ویژگی‌ها بررسی میشه. ویژگی‌هایی که ارتباط خیلی کمی با خروجی دارن یا از نظر خطی خیلی شبیه یک ویژگی دیگر هستن، گزینه خوبی برای حذف شدن هستن.
</p>

<p>
برای این کار، یک حد پایین برای همبستگی با خروجی و یک حد بالا برای همبستگی بین ویژگی‌ها در نظر گرفته شده. این مقدارها طوری انتخاب شدن که تعداد ویژگی‌ها خیلی کم نشه و در عین حال بخشی از ویژگی‌های ضعیف یا تکراری حذف بشن.
</p>

</div>

In [ ]:
def standardize_for_correlation(X):
    mean = np.mean(X, axis=0)
    std = np.std(X, axis=0)
    std = np.where(std == 0, 1.0, std)
    return (X - mean) / std


def select_features_by_correlation(
    X,
    y,
    feature_names,
    target_threshold=0.02,
    pairwise_threshold=0.92,
    min_features=35,
    max_features=65,
):
    X_std = standardize_for_correlation(X)
    y_std = (y - np.mean(y)) / (np.std(y) if np.std(y) != 0 else 1.0)

    target_corr = np.abs(np.dot(X_std.T, y_std) / (len(y) - 1))
    feature_corr = np.abs(np.dot(X_std.T, X_std) / (len(y) - 1))

    feature_order = np.argsort(target_corr)[::-1]
    selected = []

    for feature_idx in feature_order:
        if len(selected) >= max_features:
            break

        if target_corr[feature_idx] < target_threshold and len(selected) >= min_features:
            continue

        has_high_correlation = False
        for selected_idx in selected:
            if feature_corr[feature_idx, selected_idx] > pairwise_threshold:
                has_high_correlation = True
                break

        if not has_high_correlation:
            selected.append(feature_idx)

    if len(selected) < min_features:
        for feature_idx in feature_order:
            if feature_idx not in selected:
                selected.append(feature_idx)
            if len(selected) >= min_features:
                break

    selected = np.array(selected, dtype=int)

    selected_info = []
    for idx in selected:
        selected_info.append({
            "Feature": feature_names[idx],
            "Target Correlation": float(target_corr[idx]),
        })

    return selected, selected_info


selected_features, selected_info = select_features_by_correlation(
    X_train,
    y_train,
    feature_names,
    target_threshold=TARGET_CORR_THRESHOLD,
    pairwise_threshold=PAIRWISE_CORR_THRESHOLD,
    min_features=MIN_SELECTED_FEATURES,
    max_features=MAX_SELECTED_FEATURES,
)

X_train_selected = X_train[:, selected_features]
X_test_selected = X_test[:, selected_features]

print("Number of features before selection:", X_train.shape[1])
print("Number of features after selection:", X_train_selected.shape[1])

display_table(
    selected_info[:20],
    ["Feature", "Target Correlation"],
    "Top Selected Features"
)

In [ ]:
selected_model = LogisticRegression()
selected_model.fit(
    X_train_selected,
    y_train,
    epochs=EPOCHS,
    learning_rate=MAIN_LEARNING_RATE
)

selected_score = selected_model.predict_proba(X_test_selected)
selected_pred = selected_model.predict(X_test_selected)
selected_metrics = evaluate_model(y_test, selected_pred, selected_score)

part_c_rows = [
    {
        "Model": "Custom - All Features",
        "Accuracy": custom_metrics["Accuracy"],
        "Precision": custom_metrics["Precision"],
        "Recall": custom_metrics["Recall"],
        "AUC": custom_metrics["AUC"],
    },
    {
        "Model": "Custom - Selected Features",
        "Accuracy": selected_metrics["Accuracy"],
        "Precision": selected_metrics["Precision"],
        "Recall": selected_metrics["Recall"],
        "AUC": selected_metrics["AUC"],
    },
]

display_table(
    part_c_rows,
    ["Model", "Accuracy", "Precision", "Recall", "AUC"],
    "Part C - Feature Selection Result"
)

plt.figure(figsize=(8, 5))
plt.plot(selected_model.loss_history, label="Selected features")
plt.plot(custom_main_model.loss_history, label="All features")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Loss Curve Before and After Feature Selection")
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(7, 5))
selected_fpr, selected_tpr = roc_curve_manual(y_test, selected_score)

plt.plot(custom_fpr, custom_tpr, label=f"All features, AUC={auc_score(custom_fpr, custom_tpr):.3f}")
plt.plot(selected_fpr, selected_tpr, label=f"Selected features, AUC={auc_score(selected_fpr, selected_tpr):.3f}")
plt.plot([0, 1], [0, 1], linestyle="--")

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve Before and After Feature Selection")
plt.legend()
plt.grid(True)
plt.show()

<div dir="rtl" style="text-align: right; line-height: 1.9; font-family: Tahoma, Arial, sans-serif; font-size: 15px;">

<h3>۶. جمع‌بندی</h3>

<p>
در این تمرین، مدل <span dir="ltr">Logistic Regression</span> از پایه پیاده‌سازی شد و با چند مقدار مختلف برای <span dir="ltr">Learning Rate</span> آموزش دید. بعد از آموزش مدل دستی، معیارهای <span dir="ltr">Accuracy</span>، <span dir="ltr">Precision</span>، <span dir="ltr">Recall</span> و <span dir="ltr">AUC</span> برای آن گزارش شد.
</p>

<p>
در ادامه، نتیجه مدل دستی با مدل آماده <span dir="ltr">Scikit-Learn</span> مقایسه شد. در بخش آخر هم با استفاده از ماتریس همبستگی، تعدادی از ویژگی‌های کم‌اثر یا تکراری حذف شدن و مدل دوباره آموزش داده شد. این مقایسه نشان میده که کاهش ویژگی‌ها می‌تونه مدل را ساده‌تر کنه، اما اثر آن روی معیارها باید با داده تست بررسی بشه.
</p>

</div>